# Unit 4 — Resampling & Rolling Windows 🔴 CRITICAL
**Phase 2 | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

Every risk model, factor, and strategy operates at a specific **frequency** — daily, weekly, monthly.
Resampling is how you move between frequencies **without losing data integrity**.

Rolling windows are the backbone of:
- Momentum signals
- Bollinger Bands
- Z-scores
- Sharpe ratios
- **Realized volatility** (which you're computing here)

> ⚠️ **Phase 1 Weak Spot Alert**: Variance scaling (daily → annual vol) lives here.
> Get this wrong and your Sharpe ratio is meaningless.


---
## 1️⃣ Resampling — `.resample()`

`.resample(rule)` groups a time series into **frequency buckets** and applies an aggregation.

### Frequency Rules (Pandas 2.2+)
| Rule | Meaning |
|------|---------|
| `'W'` | Week-end |
| `'ME'` | Month-end ⚠️ (was `'M'` in older pandas) |
| `'QE'` | Quarter-end ⚠️ (was `'Q'`) |
| `'YE'` | Year-end ⚠️ (was `'Y'`) |
| `'D'` | Calendar day |
| `'B'` | Business day |

### Which aggregation to use?
| Data Type | Aggregation | Why |
|-----------|-------------|-----|
| Prices | `.last()` | End-of-period close price |
| Returns (log) | `.sum()` | Log returns are additive |
| Volume | `.sum()` | Total volume traded |
| Fundamentals (P/E) | `.mean()` | Average over period |
| OHLC | `.ohlc()` | Dedicated pandas method |

> 💡 This is a classic interview trick question — "when resampling daily to weekly, do you use last, mean, or sum?" — the answer is **it depends on the data type**.


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

# Download daily SPY data
spy = yf.download('SPY', start='2020-01-01', auto_adjust=True)['Close']
spy = spy.squeeze()  # Ensure it's a Series

print(f"Daily data: {len(spy)} rows")
print(spy.tail(3))


In [ ]:
# Daily → Weekly (last close of each week)
weekly = spy.resample('W').last()

# Daily → Monthly (last close of each month)
monthly = spy.resample('ME').last()

# Daily → Monthly OHLC
monthly_ohlc = spy.resample('ME').ohlc()

print(f"Weekly: {len(weekly)} rows")
print(f"Monthly: {len(monthly)} rows")
print("\nMonthly OHLC:")
print(monthly_ohlc.tail(3))


---
## 2️⃣ Rolling Windows — `.rolling()`

`.rolling(n)` creates a **sliding window** of n periods — always backward-looking by default.

### Key Parameters
| Parameter | Meaning |
|-----------|---------|
| `window` | Number of periods in the window |
| `min_periods` | Minimum observations needed (default = window size) |
| `center=False` | Default: backward-looking. `True` = centred (⚠️ look-ahead risk!) |

### Window Types
| Method | What it does | Use case |
|--------|-------------|----------|
| `.rolling(n)` | Fixed window, slides forward | Moving averages, vol estimates |
| `.expanding()` | Window grows from start | Cumulative statistics |
| `.ewm(span=n)` | Exponentially weighted | EWMA volatility (RiskMetrics) |


In [ ]:
# Log returns — always prefer for multi-period work
# Why log? Because log returns are additive: r_total = r1 + r2 + r3
# Simple returns are multiplicative — compounds errors over time
log_ret = np.log(spy / spy.shift(1)).dropna()

# Rolling volatility — different window sizes
roll_vol_20  = log_ret.rolling(20).std()  * np.sqrt(252)  # ~1 month
roll_vol_60  = log_ret.rolling(60).std()  * np.sqrt(252)  # ~3 months
roll_vol_252 = log_ret.rolling(252).std() * np.sqrt(252)  # ~1 year

# Expanding window (cumulative std from start of data)
expanding_vol = log_ret.expanding(min_periods=20).std() * np.sqrt(252)

# EWM — exponentially weighted (reacts faster to recent vol spikes)
ewm_vol = log_ret.ewm(span=20).std() * np.sqrt(252)

print("Rolling vol (last 5 rows):")
print(pd.DataFrame({
    '20d': roll_vol_20,
    '60d': roll_vol_60,
    '252d': roll_vol_252,
    'EWM': ewm_vol
}).tail(5).round(4))


---
## 3️⃣ Variance Scaling — The Formula 📐

This is the one formula you **must** have memorised for interviews.

**Variance scales linearly with time:**
```
Var(T days) = T × Var(1 day)
```

**Volatility (std dev) scales with square root of time:**
```
σ_annual  = σ_daily × √252
σ_monthly = σ_daily × √21
σ_weekly  = σ_daily × √5
```

**Why 252?** Trading days per year. Always use 252 — never mix with 365.

**Sharpe Ratio (annualised from daily):**
```
Sharpe = (mean_daily_excess_return / std_daily_return) × √252
```

> ⚠️ A common mistake: annualising the mean by ×252 but forgetting to annualise the std by ×√252.
> The Sharpe scales by √252 overall — because mean scales linearly but std scales by square root.


In [ ]:
# Variance scaling demo
daily_vol = log_ret.std()
print(f"Daily vol:   {daily_vol:.4f} ({daily_vol*100:.2f}%)")
print(f"Weekly vol:  {daily_vol * np.sqrt(5):.4f}")
print(f"Monthly vol: {daily_vol * np.sqrt(21):.4f}")
print(f"Annual vol:  {daily_vol * np.sqrt(252):.4f}")

# Verify: compute annual vol directly from annual returns
annual_ret = log_ret.resample('YE').sum()
annual_vol_direct = annual_ret.std()
annual_vol_scaled = daily_vol * np.sqrt(252)
print(f"\nDirect annual vol:  {annual_vol_direct:.4f}")
print(f"Scaled annual vol:  {annual_vol_scaled:.4f}")
print("(Should be close — not identical due to limited years of data)")


In [ ]:
# Annualised Sharpe Ratio
risk_free_daily = 0.045 / 252  # Assume 4.5% annual risk-free rate

daily_excess = log_ret - risk_free_daily
sharpe = (daily_excess.mean() / daily_excess.std()) * np.sqrt(252)
print(f"Annualised Sharpe Ratio (SPY): {sharpe:.2f}")


In [ ]:
# 🧪 OPTIONAL TASK: Volatility Term Structure
# Does short-term vol > long-term vol? (Usually yes — vol mean-reverts)

windows = {'1W': 5, '1M': 21, '3M': 63, '6M': 126, '1Y': 252}
term_structure = {}

for label, w in windows.items():
    term_structure[label] = log_ret.rolling(w).std().iloc[-1] * np.sqrt(252)

ts_df = pd.Series(term_structure, name='Realised Vol')
print("Volatility Term Structure (most recent):")
print(ts_df.round(4))

# Interpret: if 1W > 1Y, vol is currently elevated (VIX-like insight)


---
## 4️⃣ Forward-Looking Bias in Rolling Windows

`.rolling()` is **already backward-looking** by default — so no immediate bias.

The risk appears at **strategy construction time**:
- If you compute a signal today using data through today, then trade **today at open** → look-ahead bias
- You're implicitly using today's close to make a trade at today's open
- Fix: shift signals by 1 period before applying to trades

```python
# ✅ Safe: signal computed yesterday, traded today
signal = roll_vol_20.shift(1)

# ❌ Risky: signal computed today, trade also today (if prices are same-day)
signal = roll_vol_20  # used without shift
```


---
## ✅ Self-Check Questions

1. When resampling daily → weekly, should you use `.last()`, `.mean()`, or `.sum()`?
   > *Answer: Depends on data type. Prices → `.last()`. Volume → `.sum()`. Log returns → `.sum()`.*

2. What's the formula for converting daily volatility to annual?
   > *Answer: σ_annual = σ_daily × √252*

3. What's the difference between `.rolling(20).mean()` and `.resample('20D').mean()`?
   > *Answer: `.rolling(20)` slides a 20-observation window. `.resample('20D')` groups into 20-calendar-day buckets — very different, especially around weekends/holidays.*

4. How do you ensure rolling windows don't peek into the future?
   > *Answer: `.rolling()` is already backward-looking. Bias occurs when signals are used at trade time without shifting by 1 period.*

5. Why use log returns for multi-period aggregation?
   > *Answer: Log returns are additive across time: r_total = r₁ + r₂ + ... Simple returns are multiplicative, compounding errors.*

---
## 🧪 Optional Tasks

- Compute 20d, 60d, 252d rolling vol for SPY. Plot all three. Notice short-term vol spikes more violently.
- Build and plot the volatility term structure (1W, 1M, 3M, 6M, 1Y). Is short-term > long-term?
- Compare `.rolling(20).std()` vs `.ewm(span=20).std()` — which reacts faster to a spike?
- Verify variance scaling: compute annual vol two ways (directly vs scaled). They should be close.
